# Cboe 总期权活跃度来源资格

本卡只读取冻结的两个官方响应并用一次 Qlib `D.calendar` 审核覆盖，不读取未来收益。之前的看涨/看跌拆分假设已因字段缺失关闭；这里是新机制的来源资格，不是因子或组合结果。

In [ ]:
from pathlib import Path
import os, csv, io, json, hashlib
import numpy as np
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root:
        raise RuntimeError("workspace root not found")
    root = root.parent
card = root / "research/cboe_total_options_activity_source_20261022"
source_plan = json.loads((card / "plan.json").read_text())
analysis_plan = json.loads((card / "analysis_plan.json").read_text())
receipt_path = card / "retrieval_receipts.json"
receipts = json.loads(receipt_path.read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(card / "plan.json") == analysis_plan["parent_source_plan_sha256"]
assert sha(receipt_path) == analysis_plan["retrieval_receipts_sha256"]
assert sha(root / "data/qlib/soxl/qlib_store_manifest.json") == analysis_plan["calendar_manifest_sha256"]
assert sha(root / "data/qlib/soxl/calendars/day.txt") == analysis_plan["calendar_file_sha256"]
assert not (card / "run_state.json").exists(), "one frozen analysis attempt already started"
frames = {}
for item in receipts["queries"]:
    path = root / item["relative_path"]
    assert item["http_status"] == 200 and item["content_type"].startswith("text/csv")
    assert sha(path) == item["sha256"] == analysis_plan["snapshot_sha256"][item["relative_path"]]
    frame = pd.read_csv(path, dtype={"Underlying":"string", "Product Type":"string", "Exchange":"string"})
    assert list(frame.columns) == ["Trade Date", "Underlying", "Product Type", "Exchange", "Volume"]
    assert len(frame) and frame["Underlying"].eq(item["symbol"]).all()
    assert set(frame["Exchange"]).issubset(set(source_plan["parameters"]["exchanges"]))
    assert frame["Product Type"].nunique() == 1
    frame["date"] = pd.to_datetime(frame["Trade Date"], format="%Y/%m/%d")
    assert frame["date"].between(pd.Timestamp(item["startDate"]),pd.Timestamp(item["endDate"])).all()
    assert frame["Volume"].notna().all() and (frame["Volume"] >= 0).all()
    assert not frame.duplicated(["date", "Exchange"]).any()
    frames[item["symbol"]] = frame
(card / "run_state.json").write_text(json.dumps({"status":"started","analysis_plan_sha256":sha(card / "analysis_plan.json")}) + "\n")
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(root / analysis_plan["calendar_provider"]), region="us", kernels=1)
calendar = pd.DatetimeIndex(D.calendar(start_time="2019-01-02",end_time="2026-10-21",freq="day")).normalize()
assert len(calendar) and calendar.is_unique and calendar.is_monotonic_increasing
observed = {}
summary_by_symbol = {}
for symbol, frame in frames.items():
    days = pd.DatetimeIndex(sorted(frame["date"].unique()))
    expected = calendar[(calendar >= days[0]) & (calendar <= min(days[-1],calendar[-1]))]
    missing = expected.difference(days)
    extras = days.difference(calendar)
    grouped = frame.groupby("date")["Volume"].sum()
    ex_count = frame.groupby("date")["Exchange"].nunique()
    by_year = {}
    for year in sorted(set(calendar.year)):
        year_cal = calendar[calendar.year == year]
        within = year_cal[(year_cal >= days[0]) & (year_cal <= min(days[-1],calendar[-1]))]
        if len(within):
            by_year[str(year)] = {"calendar_days":len(within),"observed_days":len(within.intersection(days)),"missing_days":len(within.difference(days))}
    summary_by_symbol[symbol] = {"rows":len(frame),"observed_days":len(days),"first_date":str(days[0].date()),"last_date":str(days[-1].date()),"Qlib_calendar_last_date":str(calendar[-1].date()),"missing_Qlib_days":len(missing),"missing_first_12":[str(x.date()) for x in missing[:12]],"extra_beyond_Qlib_calendar_or_nontrading":len(extras),"extra_first_12":[str(x.date()) for x in extras[:12]],"zero_volume_days":int((grouped==0).sum()),"volume_sum":int(grouped.sum()),"four_exchange_days":int((ex_count==4).sum()),"product_type_values":sorted(set(frame["Product Type"])),"by_year":by_year}
    observed[symbol] = {"daily_total_contracts":grouped,"exchange_row_counts":ex_count,"missing_Qlib_dates":missing,"extra_dates":extras}
result = {"status":"source_coverage_qualified_only_with_PIT_gap", "source":"Cboe four-exchange historical total option volume", "plan_sha256":sha(card / "plan.json"),"analysis_plan_sha256":sha(card / "analysis_plan.json"),"retrieval_receipts_sha256":sha(receipt_path),"symbols":summary_by_symbol,"calendar_rows":len(calendar),"calendar_first_date":str(calendar[0].date()),"calendar_last_date":str(calendar[-1].date()),"last_available_date_max":max(x["last_date"] for x in summary_by_symbol.values()),"Cboe_exchange_only":True,"call_put_split_available":False,"first_public_time_qualified":False,"historical_PIT_qualified":False,"model_fits":0,"accounts":0,"market_outcome_reads":0,"D_calendar_calls":1,"D_features_calls":0,"promotion_eligible":False}
tracking = card / "qlib_recorder"
with R.start(experiment_name=source_plan["experiment"],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources", source_plan=source_plan, analysis_plan=analysis_plan, retrieval_receipts=receipts)
    rec.save_objects(artifact_path="analysis", coverage=result, observed_daily=observed, qlib_calendar=calendar)
    rec.log_metrics(source_rows=sum(x["rows"] for x in summary_by_symbol.values()), soxx_days=summary_by_symbol["SOXX"]["observed_days"], soxl_days=summary_by_symbol["SOXL"]["observed_days"], model_fits=0, accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking),recorder_local_dir=str(tracking / str(rec.experiment_id) / rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
manifest={"recorder_id":result["recorder_id"],"entries":[{"path":str(p.relative_to(card)),"sha256":sha(p),"bytes":p.stat().st_size} for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card / "verification_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
(card / "source_coverage_audit.json").write_text(json.dumps(result,indent=2)+"\n")
summary={"experiment":source_plan["experiment"],"status":result["status"],"decision":"source_available_for_separate_frozen_information_screen_after_PIT_qualification","source_coverage_audit":"source_coverage_audit.json","recorder_id":result["recorder_id"],"experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],"recorder_local_dir":result["recorder_local_dir"],"source_history_independent":False,"qualified_forward_days":0,"model_fits":0,"accounts":0,"objective_achieved":False,"artifact_policy":source_plan["artifact_policy"]}
(card / "summary.json").write_text(json.dumps(summary,indent=2)+"\n")
(card / "conclusion.md").write_text("# 中文结论\n\nCboe 官方四交易所 SOXX/SOXL 总期权成交量原字节已存档，Qlib 原生日历覆盖与缺口见 `source_coverage_audit.json`，Recorder `"+result["recorder_id"]+"`。最近可见日期不能证明首次公开时间。数据不包含看涨/看跌或开仓方向，也不覆盖其他交易所。\n\n这只证明一个非方向性总活跃度来源可进一步研究；不能解释成期权偏斜、投资者净情绪或已经验证的因子。未读取收益、拟合模型或运行账户。下一步需冻结首次公开时间与跨交易所局限，才可做独立的尾部风险信息检验。SOXX 超额与前向门槛均未通过。\n")
(card / "run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"analysis_plan_sha256":result["analysis_plan_sha256"]})+"\n")
print(json.dumps({"status":result["status"],"calendar_rows":len(calendar),"symbols":summary_by_symbol,"recorder_id":result["recorder_id"]},indent=2))

## 中文结论与下一步

SOXX/SOXL 的 Cboe 总期权量在 2019-01-02 至 2026-10-20 各覆盖 1,961 个 Qlib 交易日。年度日中位量大幅漂移（2026 年为截至 10 月 20 日的部分年度），后续信号必须用交易前可用的各标的自身滞后归一化。

纽约时间 2026-10-21 23:50 的请求没有当日行，2026-10-22 00:00:20–00:00:22 的请求已有当日行；前一日行未变。这只界定两次请求可见窗口，未证明历史首发时间或修订稳定。结论和 Recorder 身份见同卡 `conclusion.md` 与 `summary.json`。没有收益标签、拟合、账户或晋升。
